In [1]:
import torch
import gymnasium as gym
from pystk2_gymnasium import AgentSpec
import stk_actor.pystk_actor as player
import sys, os
sys.path.append(os.getcwd())

def test():
    print("🧪 TEST SIMULATION SERVEUR...")
    
    # 1. Charger poids
    state = torch.load("stk_actor/pystk_actor.pth", weights_only=True)
    print("✅ Poids chargés.")
    
    # 2. Env Difficile (Multi)
    env = gym.make("supertuxkart/multi-full-v0", render_mode="human",num_kart=4, agents=[AgentSpec(use_ai=False)])
    
    # Simuler l'adaptateur serveur
    class ServerAdapt(gym.Wrapper):
        def __init__(self, e): 
            super().__init__(e)
            self.observation_space = e.observation_space['0']
            self.action_space = e.action_space['0']
        def reset(self, **k): 
            o, i = self.env.reset(**k)
            return o['0'], i
        def step(self, a): 
            o, r, te, tr, i = self.env.step({'0': a})
            return o['0'], float(r['0']), bool(te['0']), bool(tr['0']), i
    
    env = ServerAdapt(env)
    
    # 3. Appliquer Wrappers
    for w in player.get_wrappers(): env = w(env)
    print(f"✅ Wrappers appliqués. Taille finale : {env.observation_space.shape}")

    # 4. Agent
    actor = player.get_actor(state, env.observation_space, env.action_space)
    
    # 5. Run
    from bbrl.workspace import Workspace
    ws = Workspace()
    obs, _ = env.reset()
    for t in range(20):
        ws.set("env/env_obs", t, torch.tensor(obs).unsqueeze(0))
        actor(ws, t=t)
        act = ws.get("action", t)[0].item()
        obs, _, _, _, _ = env.step(act)
        if t % 5 == 0: print(f"Step {t} OK - Action {act}")
        
    print("🎉 SUCCÈS TOTAL. PRET A ENVOYER !")
    env.close()

if __name__ == "__main__":
    test()

🧪 TEST SIMULATION SERVEUR...
✅ Poids chargés.
✅ Wrappers appliqués. Taille finale : (448,)


IndexError: invalid index to scalar variable.

In [1]:
#v2


import gymnasium as gym
import torch
import numpy as np
import sys
import os
from bbrl.workspace import Workspace
import pystk2_gymnasium 

# Ajout du path pour trouver ton module
sys.path.append(os.getcwd())

def debug_simulation():
    print("\n==================================================================")
    print("🔬 DIAGNOSTIC DE LA CHAÎNE DE WRAPPERS")
    print("==================================================================")

    # 1. Chargement du module
    try:
        from stk_actor import pystk_actor
        print("✅ Module 'stk_actor' chargé.")
    except ImportError as e:
        print(f"❌ Impossible d'importer stk_actor: {e}")
        return

    # 2. Création de l'environnement de base (Pollué comme sur le serveur)
    print("\n[1] Création de l'environnement de base...")
    try:
        # On utilise 2 karts pour éviter le bug pystk2 local
        env = gym.make("supertuxkart/simple-v0", render_mode=None, num_kart=2)
        
        # Simulation du serveur (multi-full-v0) avec des données en trop
        class ServerPollutionWrapper(gym.ObservationWrapper):
            def observation(self, obs):
                obs['team_info'] = np.zeros((5,), dtype=np.float32) # +5 dims
                obs['useless_data'] = np.random.rand(20)            # +20 dims
                return obs
        
        env = ServerPollutionWrapper(env)
        print("   -> Env de base (pollué) créé.")
        
        # Vérif taille brute (approx)
        obs_brut, _ = env.reset()
        print(f"   -> Keys brutes : {len(obs_brut.keys())} clés.")
        
    except Exception as e:
        print(f"❌ Erreur création env : {e}")
        return

    # 3. Application des Wrappers UN PAR UN (Comme le serveur)
    print("\n[2] Application des Wrappers...")
    wrappers_list = pystk_actor.get_wrappers()
    print(f"   -> Nombre de wrappers demandés : {len(wrappers_list)}")

    for i, wrapper_factory in enumerate(wrappers_list):
        print(f"\n   --- Wrapper #{i+1} ---")
        try:
            # On applique le wrapper
            env = wrapper_factory(env)
            
            # On regarde la classe du wrapper
            print(f"   Classe : {env.__class__.__name__}")
            
            # On regarde l'espace d'observation déclaré
            print(f"   Observation Space : {env.observation_space}")
            
            # TEST CRUCIAL : On fait un reset pour voir ce qui sort VRAIMENT
            obs, _ = env.reset()
            
            if isinstance(obs, dict):
                print(f"   Sortie Reset : Dict avec {len(obs)} clés")
            elif isinstance(obs, np.ndarray):
                print(f"   Sortie Reset : Array de shape {obs.shape}")
                
                # Checkpoints spécifiques
                if "FlattenWrapper" in env.__class__.__name__:
                    if obs.shape == (112,):
                        print("   ✅ FlattenWrapper : OK (112)")
                    else:
                        print(f"   ❌ FlattenWrapper : ERREUR (Reçu {obs.shape}, Attendu 112)")
                
                if "FrameStackingWrapper" in env.__class__.__name__:
                    if obs.shape == (448,):
                        print("   ✅ FrameStackingWrapper : OK (448)")
                    else:
                        print(f"   ❌ FrameStackingWrapper : ERREUR (Reçu {obs.shape}, Attendu 448)")
                        print("      -> Le stacking ne marche pas !")
            
        except Exception as e:
            print(f"   ❌ CRASH lors de l'application du wrapper #{i+1} : {e}")
            import traceback
            traceback.print_exc()
            return

    # 4. Test Final avec l'Acteur
    print("\n[3] Test Final avec l'Acteur...")
    try:
        model_path = "stk_actor/pystk_actor.pth"
        params = torch.load(model_path, map_location="cpu", weights_only=True)
        
        actor = pystk_actor.get_actor(params, env.observation_space, env.action_space)
        print("   -> Acteur chargé.")
        
        # Simulation d'un step
        obs, _ = env.reset()
        
        # La taille doit être 448 ici
        print(f"   -> Input Actor Shape : {obs.shape}")
        
        if obs.shape[0] != 448:
            print("❌ ARRÊT : L'input n'est pas 448. L'acteur va crasher.")
            return

        # Création Workspace BBRL
        workspace = Workspace()
        obs_tensor = torch.tensor(obs).unsqueeze(0).float()
        workspace.set("env/env_obs", 0, obs_tensor)
        
        # Forward
        actor(workspace, t=0)
        print("✅ SUCCESS : L'acteur a produit une action !")
        
    except Exception as e:
        print(f"❌ Erreur Acteur : {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    debug_simulation()


🔬 DIAGNOSTIC DE LA CHAÎNE DE WRAPPERS
✅ Module 'stk_actor' chargé.

[1] Création de l'environnement de base...
   -> Env de base (pollué) créé.
   -> Keys brutes : 24 clés.

[2] Application des Wrappers...
   -> Nombre de wrappers demandés : 4

   --- Wrapper #1 ---
   Classe : FeatureEngineeringWrapper
   Observation Space : Dict('attachment': Discrete(10), 'attachment_time_left': Box(0.0, inf, (1,), float32), 'aux_ticks': Box(0.0, inf, (1,), float32), 'center_path': Box(-inf, inf, (3,), float32), 'center_path_distance': Box(-inf, inf, (1,), float32), 'distance_down_track': Box(-inf, inf, (1,), float32), 'energy': Box(0.0, inf, (1,), float32), 'front': Box(-inf, inf, (3,), float32), 'items_position': Box(-inf, inf, (5, 3), float32), 'items_type': MultiDiscrete([7 7 7 7 7]), 'jumping': Discrete(2), 'karts_position': Box(-inf, inf, (5, 3), float32), 'max_steer_angle': Box(-1.0, 1.0, (1,), float32), 'paths_distance': Box(0.0, inf, (5, 2), float32), 'paths_end': Box(-inf, inf, (5, 3), fl

In [2]:
import gymnasium as gym
import torch
import numpy as np
import sys
import os
import time
from bbrl.workspace import Workspace
import pystk2_gymnasium 

# Ajout du path pour trouver ton module local
sys.path.append(os.getcwd())

# =============================================================================
# CONFIGURATION
# =============================================================================
TRACK_NAME = "zengarden" # Choisis ton circuit préféré
MODEL_PATH = "stk_actor/pystk_actor.pth"
NUM_KARTS = 2            # 2 karts pour éviter le bug pystk2 "max() empty"

# =============================================================================
# SIMULATION DE LA POLLUTION DU SERVEUR
# =============================================================================
class ServerPollutionWrapper(gym.ObservationWrapper):
    """
    Ce wrapper simule le fait que le serveur (multi-full-v0) envoie 
    beaucoup plus de données que ce que tu attends.
    Si ta 'Whitelist' dans FlattenWrapper fonctionne, le kart conduira bien.
    Sinon, ça plantera.
    """
    def __init__(self, env):
        super().__init__(env)
        
    def observation(self, obs):
        # On injecte des clés inconnues
        obs['team_info'] = np.zeros((5,), dtype=np.float32)
        obs['rescue_zone'] = np.array([1], dtype=np.int32)
        obs['garbage_data'] = np.random.rand(50) 
        return obs

def main():
    print("==================================================================")
    print("📺 VISUALISATION LOCALE (MÉTHODE SERVEUR)")
    print("==================================================================")

    # 1. Vérification du fichier .pth
    if not os.path.exists(MODEL_PATH):
        print(f"❌ ERREUR : {MODEL_PATH} introuvable.")
        print("👉 Lance d'abord 'python model_export.py'")
        return

    # 2. Chargement Sécurisé (Comme le serveur)
    print(f"🔹 Chargement des poids depuis {MODEL_PATH} (weights_only=True)...")
    try:
        params = torch.load(MODEL_PATH, map_location="cpu", weights_only=True)
        print("✅ Chargement sécurisé réussi.")
    except Exception as e:
        print(f"❌ ÉCHEC SÉCURITÉ : Ton .pth contient encore du Numpy ?\n{e}")
        return

    # 3. Création de l'environnement (visuel)
    print(f"🔹 Lancement de SuperTuxKart sur '{TRACK_NAME}'...")
    try:
        # On utilise simple-v0, mais on va le polluer juste après
        # env = gym.make(
        #     "supertuxkart/simple-v0", 
        #     render_mode="human", # ON VEUT VOIR !
        #     track=TRACK_NAME,
        #     num_kart=NUM_KARTS
        # )
        env = gym.make(
            "supertuxkart/full-v0", 
            render_mode="human", # ON VEUT VOIR !
            track="zengarden",
            num_kart=NUM_KARTS
        )
        
        # ON POLLUE L'ENVIRONNEMENT
        env = ServerPollutionWrapper(env)
        print("✅ Environnement 'pollué' (simulation serveur) prêt.")
        
    except Exception as e:
        print(f"❌ Erreur lancement STK : {e}")
        return

    # 4. Chargement de l'acteur via pystk_actor
    print("🔹 Application de tes wrappers et création de l'acteur...")
    try:
        from stk_actor import pystk_actor
        
        # A. Application des wrappers
        wrappers = pystk_actor.get_wrappers()
        for w in wrappers:
            env = w(env)
            
        print(f"ℹ️  Espace d'observation final : {env.observation_space.shape}")
        if env.observation_space.shape == (448,):
            print("✅ TAILLE CORRECTE (448) : La Whitelist a filtré la pollution !")
        else:
            print(f"❌ MAUVAISE TAILLE : {env.observation_space.shape}. Le modèle va crasher.")

        # B. Instanciation Acteur
        actor = pystk_actor.get_actor(params, env.observation_space, env.action_space)
        print("✅ Acteur prêt.")

    except Exception as e:
        print(f"❌ Erreur initialisation acteur : {e}")
        import traceback
        traceback.print_exc()
        return

    # 5. Boucle de Jeu
    print("\n🏁 DÉPART DE LA COURSE ! (Ctrl+C pour arrêter)")
    obs, _ = env.reset()
    
    try:
        while True:
            # A. Préparation Workspace
            workspace = Workspace()
            # Conversion en Tensor + Batch dimension (1, 448)
            obs_tensor = torch.tensor(obs).unsqueeze(0).float()
            workspace.set("env/env_obs", 0, obs_tensor)
            
            # B. Décision de l'Acteur
            actor(workspace, t=0)
            
            # C. Récupération Action
            action_tensor = workspace.get("action", 0)
            action_int = action_tensor.item()
            
            # D. Action dans le jeu
            obs, reward, terminated, truncated, _ = env.step(action_int)
            
            if terminated or truncated:
                print("♻️  Fin de l'épisode, Reset...")
                # obs, _ = env.reset()
                env.close()
                
            # Petit délai pour ne pas surchauffer si l'env est trop rapide (optionnel)
            # time.sleep(0.01) 

    except KeyboardInterrupt:
        print("\n🛑 Arrêt par l'utilisateur.")
    except Exception as e:
        print(f"\n❌ CRASH EN COURSE : {e}")
        import traceback
        traceback.print_exc()
    finally:
        env.close()
        print("👋 Fermeture.")

if __name__ == "__main__":
    main()

INFO: Creating pystk-2


📺 VISUALISATION LOCALE (MÉTHODE SERVEUR)
🔹 Chargement des poids depuis stk_actor/pystk_actor.pth (weights_only=True)...
✅ Chargement sécurisé réussi.
🔹 Lancement de SuperTuxKart sur 'zengarden'...
✅ Environnement 'pollué' (simulation serveur) prêt.
🔹 Application de tes wrappers et création de l'acteur...
ℹ️  Espace d'observation final : (448,)
✅ TAILLE CORRECTE (448) : La Whitelist a filtré la pollution !
✅ Acteur prêt.

🏁 DÉPART DE LA COURSE ! (Ctrl+C pour arrêter)
[ACTOR] Obs type: <class 'torch.Tensor'>
[ACTOR] Obs shape: torch.Size([1, 448])
♻️  Fin de l'épisode, Reset...

❌ CRASH EN COURSE : 
👋 Fermeture.


Traceback (most recent call last):
  File "c:\Users\PC PRO DZ\.conda\envs\rl_kart_env\lib\multiprocessing\connection.py", line 312, in _recv_bytes
    nread, err = ov.GetOverlappedResult(True)
BrokenPipeError: [WinError 109] Le canal de communication a été fermé

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "C:\Users\PC PRO DZ\AppData\Local\Temp\ipykernel_17416\159145397.py", line 131, in main
    obs, reward, terminated, truncated, _ = env.step(action_int)
  File "c:\Users\PC PRO DZ\.conda\envs\rl_kart_env\lib\site-packages\gymnasium\core.py", line 636, in step
    return self.env.step(self.action(action))
  File "c:\Users\PC PRO DZ\.conda\envs\rl_kart_env\lib\site-packages\gymnasium\core.py", line 560, in step
    observation, reward, terminated, truncated, info = self.env.step(action)
  File "c:\Users\PC PRO DZ\.conda\envs\rl_kart_env\lib\site-packages\gymnasium\core.py", line 560, in step
    observation, reward, ter

---

In [1]:
from enum import Enum
import importlib
import json
import logging
import signal
import sys
import time
import zipfile
from contextlib import contextmanager
from functools import partial
from pathlib import Path
import traceback
from tempfile import TemporaryDirectory
from typing import List, Optional
import numpy as np
import gymnasium as gym
import torch
from bbrl.agents.gymnasium import ParallelGymAgent, make_env
from bbrl.workspace import Workspace
from gymnasium import spec
from gymnasium.envs.registration import load_env_creator
from pystk2_gymnasium import AgentSpec, MonoAgentWrapperAdapter, AgentException

# --- MODIFICATION 1 : On désactive le PlotServer qui n'est pas nécessaire pour le test ---
# from master_mind.rld.stk_graph import STKLivePlotServer
STKLivePlotServer = None 

# Config de logging pour voir ce qui se passe
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

class InteractionMode(Enum):
    NONE = 0
    INTERACTIVE = 1
    MAP = 2

class TimeoutError(Exception):
    pass
i=0
@contextmanager
def timeout(seconds: float):
    # Sur Windows, SIGALRM n'existe pas. On fait un pass pour que ça marche en local.
    if sys.platform == "win32":
        yield
        return

    def timeout_handler(signum, frame):
        raise TimeoutError(f"Operation timed out after {seconds} seconds")
    
    old_handler = signal.signal(signal.SIGALRM, timeout_handler)
    signal.setitimer(signal.ITIMER_REAL, seconds)
    try:
        yield
    finally:
        signal.setitimer(signal.ITIMER_REAL, 0)
        signal.signal(signal.SIGALRM, old_handler)

def fail(message: str):
    logging.error(message)
    sys.exit(1)

@contextmanager
def sys_path(dir: Path):
    sys.path.insert(0, str(dir.absolute()))
    yield
    sys.path.pop(0)

def load_player_from_module(key: str, module_name: str, error_handling=False, name_opt: str = None):
    try:
        mod_actor = importlib.import_module(f"{module_name}.pystk_actor")
    except Exception as e:
        if error_handling:
            raise AgentException("Exception when loading module", key) from e
        raise e

    py_dir = Path(mod_actor.__file__).parent
    env_name = getattr(mod_actor, "env_name", None) or fail("No env_name in pystk_actor.py")
    
    # --- MODIFICATION POUR DEBUG ---
    # Le serveur vérifie si c'est pystk2_gymnasium.envs:STKRaceEnv.
    # On garde ça, c'est standard.
    env_spec = spec(env_name)
    
    player_name = getattr(mod_actor, "player_name", None) or fail("No player_name")
    if name_opt: player_name = name_opt[0]

    def wrappers_factory(env):
        for wrapper_spec in env_spec.additional_wrappers:
            env = load_env_creator(wrapper_spec.entry_point)(env=env, **wrapper_spec.kwargs)
        
        get_wrappers = getattr(mod_actor, "get_wrappers", lambda: [])
        for wrapper in get_wrappers():
            logging.info("Adding wrapper %s", wrapper)
            env = wrapper(env)
        return env

    get_actor = getattr(mod_actor, "get_actor", None) or fail("No get_actor")

    def actor_factory(env: gym.Env):
        pth_path = py_dir / "pystk_actor.pth"
        params = None
        if pth_path.is_file():
            # C'est ICI que le serveur applique la sécurité
            params = torch.load(
                pth_path,
                map_location=torch.device("cpu"),
                weights_only=True,
            )
        else:
            logging.warning("No pystk_actor.pth found")
        
        # Le serveur passe l'espace d'obs APRES les wrappers
        actor = get_actor(params, env.observation_space[key], env.action_space[key])
        return actor

    logging.info("Loaded player %s", player_name)
    return player_name, wrappers_factory, actor_factory

def load_player(dir: Path, key: str, file_or_module: str, error_handling=False):
    file_or_module, *name_opt = str(file_or_module).rsplit("@:", -1) # Cast path to str
    path = Path(file_or_module)
    py_dir = dir / f"player_{key}"

    # Cas Dossier Local (Ce qu'on va utiliser)
    if path.is_dir() and (path / "pystk_actor.py").is_file():
        # Sur windows les symlinks admin sont chiants, on copie ou on ajoute au path
        # Ici on hack un peu pour le local : on ajoute le parent au path
        with sys_path(path.parent):
            return load_player_from_module(key, path.name, name_opt=name_opt, error_handling=error_handling)

    # Cas ZIP... (Code original omis pour brièveté, inutile ici)
    return load_player_from_module(key, file_or_module, error_handling=error_handling, name_opt=name_opt)

def get_action(workspace: Workspace, t: int):
    name = "action"
    if name in workspace.variables:
        action = workspace.get(name, t)
    else:
        action = {} # Gestion dict...
        # Simplifié pour ton cas (tu retournes un tenseur)
    return action

def dict_slice(k: int, object):
    if isinstance(object, dict):
        return {key: dict_slice(k, value) for key, value in object.items()}
    return object[k]
i = 0
@torch.no_grad()
def race(i, hide: bool, num_karts: Optional[int], zip_files: List[Path], interaction=InteractionMode.NONE):
    server = None
    num_karts = num_karts or len(zip_files)

    # On utilise un dossier temp pour simuler l'extraction
    with TemporaryDirectory() as dir:
        dir = Path(dir)
        logging.info("Loading the agents")
        agent_factories = {}
        wrapper_factories = {}
        agents_spec = []
        player_names = []
        
        for agent_ix, zip_file in enumerate(zip_files):
            key = str(agent_ix)
            player_name, wrappers_factory, agent_factory = load_player(
                dir, key, zip_file, error_handling=True
            )
            player_names.append(player_name)
            agent_factories[key] = agent_factory
            wrapper_factories[key] = wrappers_factory
            agents_spec.append(AgentSpec(name=player_name))

        # --- LE TEST ULTIME : multi-full-v0 ---
        logging.info("Creating environment 'supertuxkart/multi-full-v0'")
        n_agents = len(agents_spec)
        env = make_env(
            "supertuxkart/multi-full-v0", # C'EST LUI LE COUPABLE !
            track="fortmagma",
            render_mode=None if hide else "human",
            num_kart=num_karts,
            agents=agents_spec,
            wrappers=[
                partial(
                    MonoAgentWrapperAdapter,
                    keep_original=interaction != InteractionMode.NONE,
                    wrapper_factories=wrapper_factories,
                )
            ],
        )

        agents = []
        for key, agent_factory in agent_factories.items():
            try:
                # C'est ici que ça crashait (112 vs 117)
                agent = agent_factory(env)
                agents.append(agent)
                print(f"✅ Agent {key} initialisé avec succès !")
            except Exception as e:
                print(f"❌ CRASH lors de l'init de l'agent {key} : {e}")
                raise e

        for agent in agents:
            agent.eval()

        workspaces = [Workspace() for _ in range(n_agents)]
        logging.info("Starting a race")

        done = False
        obs, _ = env.reset()
        t = 0
        
        try:
            while not done:
                actions = {}
                for ix in range(n_agents):
                    key = str(ix)
                    # BBRL Formatting
                    obs_agent = ParallelGymAgent._format_frame(obs[key])
                    for var_key, var_value in obs_agent.items():
                        workspaces[ix].set(f"env/{var_key}", t, var_value)

                    # --- EXECUTION AGENT ---
                    try:
                        agents[ix](workspaces[ix], t=t)
                        action = get_action(workspaces[ix], t=t)
                        
                        # Vérif que l'action est bien sortie
                        if action is None:
                            print(f"❌ Pas d'action générée à t={t}")
                    except Exception as e:
                        print(f"❌ CRASH lors du choix de l'action à t={t} : {e}")
                        raise e

                    if isinstance(action, dict):
                        action = dict_slice(0, action)
                    else:
                        action = action[0]
                    
                    actions[key] = action

                obs, reward, terminated, truncated, info = env.step(actions)
                i+=1
                if i>200:
                    break
                # print("action: ", action)
                
                
                done = terminated or truncated
                t += 1
                
                # Petit print pour montrer que ça tourne
                print(f"\rStep {t} OK - Action Joueur: {actions['0']}", end="")
                
        except KeyboardInterrupt:
            print("\nArrêt manuel.")
        finally:
            env.close()
            print("\nFin de la simulation.")

# =============================================================================
# MAIN : LANCEMENT DU TEST LOCAL
# =============================================================================
if __name__ == "__main__":
    # Chemin vers ton dossier contenant pystk_actor.py
    # Si le script est à la racine, c'est juste "stk_actor"
    agent_path = Path("stk_actor")
    
    if not agent_path.exists():
        print("❌ Dossier stk_actor introuvable. Vérifie le chemin.")
        sys.exit(1)

    print("🚀 Lancement de la simulation EXACTE du serveur...")
    i=0
    race(i,
        hide=False,       # On veut voir le kart
        num_karts=4,      # 2 karts pour éviter le bug pystk2
        zip_files=[agent_path]
    )

INFO: Loading the agents
INFO: Loaded player Team_PPO_Final
INFO: Creating environment 'supertuxkart/multi-full-v0'
INFO: Creating pystk-1


🚀 Lancement de la simulation EXACTE du serveur...


INFO: Adding wrapper <function get_wrappers.<locals>.<lambda> at 0x00000229E55ED360>
INFO: Adding wrapper <function get_wrappers.<locals>.<lambda> at 0x00000229E55EC550>
INFO: Starting a race


✅ Agent 0 initialisé avec succès !
[ACTOR] Obs type: <class 'torch.Tensor'>
[ACTOR] Obs shape: torch.Size([1, 448])
Step 200 OK - Action Joueur: 12
Fin de la simulation.


In [1]:
from enum import Enum
import importlib
import json
import logging
import signal
import sys
import time
from contextlib import contextmanager
from functools import partial
from pathlib import Path
import traceback
from tempfile import TemporaryDirectory
from typing import List, Optional
import numpy as np
import gymnasium as gym
import torch
from bbrl.agents.gymnasium import ParallelGymAgent, make_env
from bbrl.workspace import Workspace
from gymnasium import spec
from gymnasium.envs.registration import load_env_creator
from pystk2_gymnasium import AgentSpec, MonoAgentWrapperAdapter, AgentException

# Désactivation Plot
STKLivePlotServer = None 
logging.basicConfig(level=logging.INFO)

class InteractionMode(Enum):
    NONE = 0

class TimeoutError(Exception):
    pass

@contextmanager
def timeout(seconds: float):
    if sys.platform == "win32": yield; return
    def timeout_handler(signum, frame): raise TimeoutError(f"Timeout")
    old_handler = signal.signal(signal.SIGALRM, timeout_handler)
    signal.setitimer(signal.ITIMER_REAL, seconds)
    try: yield
    finally: signal.setitimer(signal.ITIMER_REAL, 0); signal.signal(signal.SIGALRM, old_handler)

def fail(message: str):
    logging.error(message)
    sys.exit(1)

@contextmanager
def sys_path(dir: Path):
    sys.path.insert(0, str(dir.absolute()))
    yield
    sys.path.pop(0)

def load_player_from_module(key: str, module_name: str, error_handling=False, name_opt: str = None):
    try:
        mod_actor = importlib.import_module(f"{module_name}.pystk_actor")
    except Exception as e:
        raise e

    py_dir = Path(mod_actor.__file__).parent
    env_name = getattr(mod_actor, "env_name", None) or fail("No env_name")
    env_spec = spec(env_name)
    player_name = getattr(mod_actor, "player_name", None) or fail("No player_name")

    def wrappers_factory(env):
        # 1. Wrappers Env spec
        for wrapper_spec in env_spec.additional_wrappers:
            env = load_env_creator(wrapper_spec.entry_point)(env=env, **wrapper_spec.kwargs)
        # 2. Wrappers Agent
        get_wrappers = getattr(mod_actor, "get_wrappers", lambda: [])
        for wrapper in get_wrappers():
            logging.info("Adding wrapper %s", wrapper)
            env = wrapper(env)
        return env

    get_actor = getattr(mod_actor, "get_actor", None) or fail("No get_actor")

    def actor_factory(env: gym.Env):
        pth_path = py_dir / "pystk_actor.pth"
        params = torch.load(pth_path, map_location="cpu", weights_only=True)
        actor = get_actor(params, env.observation_space[key], env.action_space[key])
        return actor

    return player_name, wrappers_factory, actor_factory

def load_player(dir: Path, key: str, file_or_module: str, error_handling=False):
    file_or_module = str(file_or_module)
    path = Path(file_or_module)
    if path.is_dir():
        with sys_path(path.parent):
            return load_player_from_module(key, path.name, error_handling=error_handling)
    return load_player_from_module(key, file_or_module, error_handling=error_handling)

def get_action(workspace: Workspace, t: int):
    name = "action"
    if name in workspace.variables: return workspace.get(name, t)
    return {}

def dict_slice(k: int, object):
    if isinstance(object, dict): return {key: dict_slice(k, value) for key, value in object.items()}
    return object[k]

@torch.no_grad()
def race(num_karts: int, zip_files: List[Path]):
    with TemporaryDirectory() as dir:
        dir = Path(dir)
        agent_factories, wrapper_factories, agents_spec = {}, {}, []
        
        for agent_ix, zip_file in enumerate(zip_files):
            key = str(agent_ix)
            player_name, wrappers_factory, agent_factory = load_player(dir, key, zip_file)
            agent_factories[key] = agent_factory
            wrapper_factories[key] = wrappers_factory
            agents_spec.append(AgentSpec(name=player_name))

        # ENVIRONNEMENT DU SERVEUR
        print("\n--- CRÉATION ENVIRONNEMENT SERVEUR (MULTI-FULL) ---")
        env = make_env(
            "supertuxkart/multi-full-v0",
            render_mode="human",
            num_kart=num_karts,
            agents=agents_spec,
            wrappers=[partial(MonoAgentWrapperAdapter, wrapper_factories=wrapper_factories)],
        )

        agents = []
        for key, agent_factory in agent_factories.items():
            agents.append(agent_factory(env))

        for agent in agents: agent.eval()
        workspaces = [Workspace() for _ in range(len(agents_spec))]

        print("\n--- DÉBUT DE LA COURSE ---")
        obs, _ = env.reset()
        
        try:
            for t in range(500): # 500 steps
                actions = {}
                for ix in range(len(agents_spec)):
                    key = str(ix)
                    obs_agent = ParallelGymAgent._format_frame(obs[key])
                    for k, v in obs_agent.items(): workspaces[ix].set(f"env/{k}", t, v)

                    agents[ix](workspaces[ix], t=t)
                    action = get_action(workspaces[ix], t=t)
                    
                    if isinstance(action, dict): action = dict_slice(0, action)
                    else: action = action[0]
                    
                    actions[key] = action

                obs, reward, _, _, _ = env.step(actions)
                print(f"\rStep {t} OK", end="")
        except KeyboardInterrupt:
            pass
        finally:
            env.close()

if __name__ == "__main__":
    agent_path = Path("stk_actor")
    if not agent_path.exists(): sys.exit("Dossier introuvable")
    
    # On lance avec 2 karts pour que ça tourne
    race(num_karts=2, zip_files=[agent_path])

INFO:root:Creating pystk-1



--- CRÉATION ENVIRONNEMENT SERVEUR (MULTI-FULL) ---


INFO:root:Adding wrapper <function get_wrappers.<locals>.<lambda> at 0x00000254965C1240>
INFO:root:Adding wrapper <function get_wrappers.<locals>.<lambda> at 0x00000254965C1090>



--- DÉBUT DE LA COURSE ---

[WRAPPER SPY] Frame 0
   -> Center Path: [-1.4867724  -0.05686909 -0.01498365]
   ✅ INFO : Les valeurs sont petites. PolarObservations semble actif.


AgentException: 'tuple' object has no attribute 'size'

In [4]:
import pystk2_gymnasium

pystk2_gymnasium.__version__



'0.0.0'

In [3]:

from pystk_gymnasium import AgentSpec

agents = [
    AgentSpec(use_ai=True, name="Yin Team", camera_mode=CameraMode.ON),
    AgentSpec(use_ai=True, name="Yang Team", camera_mode=CameraMode.ON),
]


wrappers = [
    partial(MonoAgentWrapperAdapter, wrapper_factories={
        "0": lambda env: ConstantSizedObservations(env),
        "1": lambda env: PolarObservations(ConstantSizedObservations(env)),
        "2": lambda env: PolarObservations(ConstantSizedObservations(env))
    }),
]

make_stkenv = partial(
    make_env,
    "supertuxkart/multi-full-v0",
    render_mode="human",
    num_kart=5,
    agents=agents,
    wrappers=wrappers
)

ModuleNotFoundError: No module named 'pystk_gymnasium'

In [2]:
import pystk2_gymnasium.wrappers as agent
print(agent.__file__)




c:\Users\PC PRO DZ\.conda\envs\rl_kart_env\lib\site-packages\pystk2_gymnasium\wrappers.py


In [1]:
import gymnasium as gym
import torch
import numpy as np
import os
from bbrl.workspace import Workspace

# On importe ton code pour charger l'acteur correctement
from stk_actor.pystk_actor import get_actor, env_name

import gymnasium as gym
import pystk2_gymnasium  # <--- INDISPENSABLE



reawrds =[]


def test_local():
    print("--- 🏎️  LANCEMENT DIRECT (Sans Master-Mind) 🏎️ ---")
    
    # 1. Création de l'environnement avec fenêtre
    try:
        # render_mode="human" est crucial pour voir la fenêtre
        env = gym.make(env_name, render_mode="human")
    except Exception as e:
        print(f"❌ Erreur Gym: {e}")
        return

    # 2. Wrapper Aplatisseur (comme sur le serveur)
    from gymnasium.wrappers import FlattenObservation
    env = FlattenObservation(env)
    
    # 3. Chargement du cerveau
    path_pth = "stk_actor/pystk_actor.pth"
    if not os.path.exists(path_pth):
        print(f"❌ ERREUR: Je ne trouve pas {path_pth}")
        return
    
    print(f"Chargement des poids depuis {path_pth}...")
    state_dict = torch.load(path_pth, map_location="cpu")

    # 4. Création de l'agent
    actor = get_actor(state_dict, env.observation_space, env.action_space)
    
    # 5. Boucle de jeu
    obs, _ = env.reset()
    workspace = Workspace()
    
    print("\n🟢 C'EST PARTI ! Regarde la fenêtre SuperTuxKart.")
    
    t = 0
    done = False
    try:
        while not done:
            # Conversion Obs -> Tensor Batch
            obs_tensor = torch.tensor(obs, dtype=torch.float32)
            if obs_tensor.dim() == 1:
                obs_tensor = obs_tensor.unsqueeze(0)
            
            # BBRL Workspace
            workspace.set("env/env_obs", t, obs_tensor)
            
            # Action
            actor(workspace, t=t)
            action = workspace.get("action", t).squeeze(0).numpy()
            
            # Step
            obs, reward, terminated, truncated, _ = env.step(action)
            reawrds.append(reward)
            done = terminated or truncated
            t += 1
            
    except KeyboardInterrupt:
        print("Arrêt utilisateur.")
    finally:
        env.close()

if __name__ == "__main__":
    test_local()

--- 🏎️  LANCEMENT DIRECT (Sans Master-Mind) 🏎️ ---
Chargement des poids depuis stk_actor/pystk_actor.pth...


C:\Users\PC PRO DZ\AppData\Local\Temp\ipykernel_20544\1375774121.py:40: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(path_pth, map_location="cpu")



🟢 C'EST PARTI ! Regarde la fenêtre SuperTuxKart.


In [5]:
np.mean(reawrds)

np.float64(-0.03022229410807294)

In [2]:
import gymnasium as gym
import pystk2_gymnasium  # Indispensable pour que Gym trouve SuperTuxKart
import torch
import numpy as np
import os
import time
from bbrl.workspace import Workspace

# --- IMPORTATION DU CODE DE SOUMISSION ---
# C'est ici qu'on teste si ton code stk_actor/ marche vraiment
try:
    from stk_actor.pystk_actor import get_actor, get_wrappers, env_name
    print("✅ Importation de stk_actor réussie.")
except ImportError as e:
    print(f"❌ ERREUR CRITIQUE : Impossible d'importer stk_actor : {e}")
    print("Vérifie que tu es bien à la racine du projet et que PYTHONPATH=. est set.")
    exit(1)
rewards = []

def run_verification(rewards):
    print("\n--- 🕵️  VERIFICATION DU CODE DE SOUMISSION (LOCAL) 🕵️ ---")

    # 1. Création de l'environnement (avec fenêtre pour voir)
    try:
        env = gym.make(env_name, render_mode="human", num_kart=4)
    except Exception as e:
        print(f"❌ Erreur lors de la création de l'environnement : {e}")
        return

    # 2. Application des Wrappers (ceux définis dans pystk_actor.py)
    # Normalement, il n'y a que FlattenObservation ici.
    wrappers = get_wrappers()
    for w in wrappers:
        env = w(env)
    print("✅ Wrappers appliqués.")

    # 3. Chargement du fichier de poids (.pth)
    model_path = "stk_actor/pystk_actor.pth"
    if not os.path.exists(model_path):
        print(f"❌ ERREUR : Le fichier {model_path} est introuvable !")
        return
    
    print(f"🔹 Chargement des poids depuis {model_path}...")
    try:
        state_dict = torch.load(model_path, map_location="cpu")
    except Exception as e:
        print(f"❌ Le fichier .pth semble corrompu : {e}")
        return

    # 4. Création de l'Agent (Le Cerveau)
    # C'est ici que la classe Actor de actors.py est instanciée
    try:
        actor = get_actor(state_dict, env.observation_space, env.action_space)
        print("✅ Agent créé avec succès (FrameStacking interne activé).")
    except Exception as e:
        print(f"❌ Erreur dans get_actor (vérifie actors.py) : {e}")
        return

    # 5. Simulation de la course
    print("\n🟢 DÉBUT DE LA COURSE DE TEST")
    print("Observe bien le kart :")
    print("  - S'il tourne en rond -> Problème d'ordre des actions.")
    print("  - S'il ne bouge pas -> Problème de poids (zéros).")
    print("  - S'il conduit bien -> TU ES PRÊT À PUSH !")
    
    obs, _ = env.reset()
    workspace = Workspace()
    
    t = 0
    total_reward = 0
    done = False
    
    try:
        while not done:
            # --- Simulation de ce que fait BBRL sur le serveur ---
            
            # A. Préparation de l'observation
            # On convertit en Tensor et on ajoute la dimension Batch (1, 154)
            obs_tensor = torch.tensor(obs, dtype=torch.float32)
            if obs_tensor.dim() == 1:
                obs_tensor = obs_tensor.unsqueeze(0)
            
            # B. Injection dans le Workspace
            workspace.set("env/env_obs", t, obs_tensor)
            
            # C. L'Agent réfléchit (Appel de actors.py -> forward)
            # C'est là que ton code "History" et "Torch.clamp" s'exécute
            actor(workspace, t=t)
            
            # D. Récupération de l'action
            action_tensor = workspace.get("action", t)
            action = action_tensor.squeeze(0).numpy() # (1, ...) -> (...)
            
            # E. Exécution dans le jeu
            obs, reward, terminated, truncated, _ = env.step(action)
            rewards.append(reward)
            
            total_reward += reward
            done = terminated or truncated
            t += 1
            
            # Petit affichage pour dire que ça tourne
            if t % 100 == 0:
                print(f"   Step {t} | Reward cumulée: {total_reward:.2f}")

    except KeyboardInterrupt:
        print("🛑 Arrêt manuel.")
    except Exception as e:
        print(f"❌ ERREUR PENDANT LA COURSE : {e}")
        import traceback
        traceback.print_exc()
    finally:
        env.close()
        print(f"\n🏁 Fin du test. Reward finale : {total_reward:.2f}")

if __name__ == "__main__":
    rewards_for_all = []
    
    for track in range(1):
        rewards = []
        run_verification(rewards)
        print(np.mean(rewards))
        rewards_for_all.append(np.mean(rewards))
        
        print(f" avg rew : {np.mean(rewards_for_all)}")

✅ Importation de stk_actor réussie.

--- 🕵️  VERIFICATION DU CODE DE SOUMISSION (LOCAL) 🕵️ ---
✅ Wrappers appliqués.
🔹 Chargement des poids depuis stk_actor/pystk_actor.pth...
✅ Agent créé avec succès (FrameStacking interne activé).

🟢 DÉBUT DE LA COURSE DE TEST
Observe bien le kart :
  - S'il tourne en rond -> Problème d'ordre des actions.
  - S'il ne bouge pas -> Problème de poids (zéros).
  - S'il conduit bien -> TU ES PRÊT À PUSH !


C:\Users\PC PRO DZ\AppData\Local\Temp\ipykernel_10796\2948789690.py:45: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(model_path, map_location="cpu")

   Step 100 | Reward cumulée: 48.33
   Step 200 | Reward cumulée: 39.02
   Step 300 | Reward cumulée: 31.67
🛑 Arrêt manuel.

🏁 Fin du test. Reward finale : 20.77
0.054945861977874925
 avg rew : 0.054945861977874925


In [3]:
import gymnasium as gym
import torch
import numpy as np
import sys
import os
from bbrl.workspace import Workspace
import pystk2_gymnasium # Nécessaire pour charger les envs STK

# Ajout du path pour trouver ton module
sys.path.append(os.getcwd())

def test_server_simulation():
    print("==================================================================")
    print("🕵️  SIMULATION DU SERVEUR D'ÉVALUATION")
    print("==================================================================")

    # 1. TEST DE SÉCURITÉ (Torch Load)
    # Le serveur utilise weights_only=True. Si tu as des Numpy arrays dans le .pth, ça crash.
    print("\n[1] Test de chargement sécurisé (weights_only=True)...")
    model_path = "stk_actor/pystk_actor.pth"
    
    if not os.path.exists(model_path):
        print(f"❌ ERREUR : Le fichier {model_path} n'existe pas !")
        return

    try:
        # C'est LA ligne qui fait planter le serveur si mal configuré
        params = torch.load(model_path, map_location="cpu", weights_only=True)
        print("✅ SUCCESS : Le fichier .pth est sécurisé (contient des Tensors).")
    except Exception as e:
        print("❌ CRASH SÉCURITÉ : Ton .pth contient probablement du Numpy.")
        print(f"   Erreur : {e}")
        return

    # 2. CRÉATION DE L'ENVIRONNEMENT (Le plus dur : multi-full-v0)
    # Le serveur utilise multi-full-v0 qui a beaucoup plus de clés que simple-v0.
    print("\n[2] Création de l'environnement serveur (multi-full-v0)...")
    try:
        # On essaie de charger le 'vrai' env du serveur
        env = gym.make("supertuxkart/multi-full-v0", render_mode=None, num_kart=1)
        print("   -> Environnement 'multi-full-v0' chargé.")
    except Exception as e:
        print("   ⚠️  Impossible de charger 'multi-full-v0' localement.")
        print("   -> Fallback sur 'simple-v0' avec injection de fausses clés parasites.")
        env = gym.make("supertuxkart/simple-v0", render_mode=None, num_kart=1)
        
        # On va créer un wrapper méchant qui ajoute des données inutiles pour piéger ton code
        class EvilServerWrapper(gym.ObservationWrapper):
            def observation(self, obs):
                # Le serveur ajoute souvent ça
                obs['team_info'] = np.zeros((5,), dtype=np.float32)
                obs['rescue_zone'] = np.array([1], dtype=np.int32)
                obs['magic_variable'] = np.random.rand(10) # Variable inconnue
                return obs
        env = EvilServerWrapper(env)
        print("   -> Environnement 'simple-v0' pollué (simulation 117+ dims) prêt.")

    # 3. CHARGEMENT DE TON CODE
    print("\n[3] Chargement de tes wrappers et de l'acteur...")
    try:
        from stk_actor import pystk_actor
        
        # A. Wrappers
        wrappers = pystk_actor.get_wrappers()
        print(f"   -> {len(wrappers)} wrappers trouvés.")
        
        # B. Application des wrappers
        for i, w in enumerate(wrappers):
            env = w(env)
            # print(f"      Wrapper {i+1} appliqué. Obs space: {env.observation_space}")

        print(f"   -> Espace d'observation FINAL : {env.observation_space}")
        
        # VÉRIFICATION CRITIQUE DE LA TAILLE
        # On attend (448,) car 112 * 4 frames
        if env.observation_space.shape == (448,):
            print("✅ TAILLE PARFAITE : 448 dimensions.")
        else:
            print(f"⚠️  ATTENTION : Taille obtenue {env.observation_space.shape}. Ton modèle attend 448.")
            print("    Si ce n'est pas 448, ça va probablement crasher à l'étape suivante.")

        # C. Instanciation de l'acteur
        actor = pystk_actor.get_actor(params, env.observation_space, env.action_space)
        print("✅ Acteur instancié avec succès.")

    except Exception as e:
        print(f"❌ ERREUR lors du chargement de l'agent : {e}")
        import traceback
        traceback.print_exc()
        return

    # 4. SIMULATION D'UNE COURSE (Boucle BBRL)
    print("\n[4] Simulation d'un pas de temps (Forward)...")
    try:
        # Reset env
        obs, _ = env.reset()
        
        # Création Workspace BBRL (Simule ParallelGymAgent)
        workspace = Workspace()
        
        # On simule le formatage BBRL : Ajout dimension Batch + Conversion Tensor
        # Obs actuelle : (448,) -> On veut (1, 448) (Batch size 1)
        obs_tensor = torch.tensor(obs).unsqueeze(0).float()
        
        # Injection dans le workspace
        workspace.set("env/env_obs", 0, obs_tensor)
        
        # Exécution de l'acteur
        print("   -> Appel de actor(workspace, t=0)...")
        actor(workspace, t=0)
        
        # Récupération de l'action
        action = workspace.get("action", 0)
        print(f"✅ ACTION GÉNÉRÉE : {action}")
        print(f"   Shape: {action.shape} (Doit être [1])")
        
        # Vérification si l'action est valide pour l'env
        # Ton wrapper attend un int, mais BBRL sort un Tensor([int])
        action_int = action.item()
        print(f"   Action (int) : {action_int}")
        
        # Step dans l'env pour être sûr que la conversion d'action marche
        print("   -> Env.step() avec l'action...")
        _, _, _, _, _ = env.step(action_int)
        print("✅ Env.step() réussi !")

    except Exception as e:
        print(f"❌ CRASH PENDANT L'EXÉCUTION : {e}")
        import traceback
        traceback.print_exc()
        return

    print("\n==================================================================")
    print("🎉 RÉSULTAT FINAL : TOUT SEMBLE OK !")
    print("Si ce script passe, ton code a 99% de chances de marcher sur le serveur.")
    print("==================================================================")

if __name__ == "__main__":
    test_server_simulation()

🕵️  SIMULATION DU SERVEUR D'ÉVALUATION

[1] Test de chargement sécurisé (weights_only=True)...
✅ SUCCESS : Le fichier .pth est sécurisé (contient des Tensors).

[2] Création de l'environnement serveur (multi-full-v0)...
   ⚠️  Impossible de charger 'multi-full-v0' localement.
   -> Fallback sur 'simple-v0' avec injection de fausses clés parasites.
   -> Environnement 'simple-v0' pollué (simulation 117+ dims) prêt.



[3] Chargement de tes wrappers et de l'acteur...
   -> 3 wrappers trouvés.
❌ ERREUR lors du chargement de l'agent : max() arg is an empty sequence


Traceback (most recent call last):
  File "C:\Users\PC PRO DZ\AppData\Local\Temp\ipykernel_10796\3876998881.py", line 69, in test_server_simulation
    env = w(env)
  File "c:\Users\Amine\Bureau\Programme_Python\pystk2-project-template\stk_actor\pystk_actor.py", line 22, in <lambda>
    lambda env: DiscreteActionWrapper(env),
  File "c:\Users\Amine\Bureau\Programme_Python\pystk2-project-template\stk_actor\wrappers.py", line 162, in __init__
    dummy_obs, _ = self.env.reset()
  File "c:\Users\Amine\Bureau\Programme_Python\pystk2-project-template\stk_actor\wrappers.py", line 19, in reset
    obs, info = self.env.reset(**kwargs)
  File "c:\Users\PC PRO DZ\.conda\envs\rl_kart_env\lib\site-packages\gymnasium\core.py", line 553, in reset
    obs, info = self.env.reset(seed=seed, options=options)
  File "c:\Users\PC PRO DZ\.conda\envs\rl_kart_env\lib\site-packages\gymnasium\core.py", line 553, in reset
    obs, info = self.env.reset(seed=seed, options=options)
  File "c:\Users\PC PRO DZ\.con

In [38]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np

class DiscreteActionWrapper(gym.Wrapper):
    def __init__(self, env):
        super().__init__(env)
        
        # --- 1. CONFIGURATION STRICTE ---
        # On définit exactement la forme attendue pour chaque clé "variable".
        # Basé sur simple-v0 qui limite à 5 éléments.
        # Format: 'Nom': (Nombre_Lignes_Max, Nombre_Colonnes)
        self.target_shapes = {
            'items_position': (5, 3),  # 5 items, x,y,z
            'items_type': (5, 1),      # 5 types (souvent un vecteur colonne ou plat)
            'karts_position': (5, 3),  # 5 karts
            'paths_distance': (5, 2),
            'paths_end': (5, 3),
            'paths_start': (5, 3),
            'paths_width': (5, 1),
        }
        
        # Liste des clés scalaires/fixes à garder
        self.fixed_keys = sorted([
            'attachment', 'attachment_time_left', 'aux_ticks', 
            'center_path', 'center_path_distance', 'distance_down_track', 
            'energy', 'front', 'jumping', 'max_steer_angle', 'phase', 
            'powerup', 'shield_time', 'skeed_factor', 'velocity',
            
            # Tes features custom
            'feat_speed', 'feat_dist_center', 'feat_angle_center', 
            'feat_future_risk', 'feat_lookahead_angle', 'feat_curve_intensity', 
            'feat_skeed', 'feat_air', 'feat_item_angle', 'feat_item_detected', 
            'feat_off_track'
        ])
        
        # On combine tout pour la boucle principale
        self.all_keys = sorted(self.fixed_keys + list(self.target_shapes.keys()))

        # --- 2. CALCUL DE LA TAILLE FIXE ---
        # On fait un test à blanc pour calculer la taille exacte UNE FOIS pour toutes.
        dummy_obs, _ = self.env.reset()
        flat_obs = self._flatten_obs(dummy_obs, debug=True)
        self.flat_size = flat_obs.shape[0]
        
        print(f"🔒 DiscreteActionWrapper: Taille FORCÉE et FIXÉE à {self.flat_size}")
        
        # On impose cette taille dans l'espace d'observation
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(self.flat_size,), dtype=np.float32
        )

        # Liste d'actions (inchangée)
        self.actions_list = [
            (0.0, 1.0, 0, 0, 0, 0, 0), (-0.6, 1.0, 0, 0, 0, 0, 0), (0.6, 1.0, 0, 0, 0, 0, 0),
            (-1.0, 1.0, 0, 1, 0, 0, 0), (1.0, 1.0, 0, 1, 0, 0, 0), (-0.5, 1.0, 0, 1, 0, 0, 0),
            (0.5, 1.0, 0, 1, 0, 0, 0), (0.0, 0.3, 0, 0, 0, 0, 0), (-0.6, 0.3, 0, 0, 0, 0, 0),
            (0.6, 0.3, 0, 0, 0, 0, 0), (0.0, 0.0, 1, 0, 0, 0, 0), (-1.0, 0.0, 1, 0, 0, 0, 0),
            (1.0, 0.0, 1, 0, 0, 0, 0), (0.0, 1.0, 0, 0, 1, 0, 0), (0.0, 1.0, 0, 0, 0, 1, 0),
        ]
        self.action_space = spaces.Discrete(len(self.actions_list))

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        return self._flatten_obs(obs), info

    def step(self, action_idx):
        vals = self.actions_list[action_idx]
        game_action = {
            'steer': np.array([vals[0]], dtype=np.float32),
            'acceleration': np.array([vals[1]], dtype=np.float32),
            'brake': vals[2], 'drift': vals[3], 'fire': vals[4], 'nitro': vals[5], 'rescue': vals[6]
        }
        obs, reward, terminated, truncated, info = self.env.step(game_action)
        return self._flatten_obs(obs), reward, terminated, truncated, info

    def _flatten_obs(self, obs, debug=False):
        flat_list = []
        
        for key in self.all_keys:
            if key in obs:
                val = obs[key]
                
                # --- CAS 1: TABLEAUX VARIABLES (Items, Paths, etc.) ---
                if key in self.target_shapes:
                    target_rows, target_cols = self.target_shapes[key]
                    
                    # Convertir en numpy float32
                    arr = np.array(val, dtype=np.float32)
                    
                    # Gérer items_type qui peut être plat (N,) au lieu de (N, 1)
                    if len(arr.shape) == 1:
                        arr = arr.reshape(-1, 1)
                    
                    # A. TRONCATURE (Si trop grand)
                    if arr.shape[0] > target_rows:
                        arr = arr[:target_rows, :]
                        
                    # B. PADDING (Si trop petit)
                    elif arr.shape[0] < target_rows:
                        # On crée des zéros
                        padding = np.zeros((target_rows - arr.shape[0], arr.shape[1]), dtype=np.float32)
                        arr = np.vstack([arr, padding])
                        
                    flat_list.append(arr.flatten())

                # --- CAS 2: CLÉS FIXES (Velocity, etc.) ---
                else:
                    if isinstance(val, (int, float, np.number)):
                        flat_list.append([float(val)])
                    else:
                        flat_list.append(np.array(val, dtype=np.float32).flatten())
            else:
                # Si une clé manque (ex: feat_... pas encore calculé ou bug env)
                # On remplit avec des zéros selon la spec pour ne pas casser la taille
                if debug: print(f"⚠️  Missing Key remplacée par 0: {key}")
                if key in self.target_shapes:
                    r, c = self.target_shapes[key]
                    flat_list.append(np.zeros(r*c, dtype=np.float32))
                else:
                    # Pour les scalaires/vecteurs fixes, difficile de deviner la taille exacte 
                    # sans spec, mais généralement c'est 1 ou 3.
                    # Pour l'instant on ignore pour ne pas polluer, ou on met un 0 par défaut.
                    # Mieux vaut que le FeatureWrapper fasse bien son job.
                    pass

        return np.concatenate(flat_list).astype(np.float32)

In [39]:
import gymnasium as gym
from pystk2_gymnasium import AgentSpec
import numpy as np
import sys
import os

# Import tes wrappers locaux
try:
    print("1")
    from my_stk_agent.stk_actor.wrappers_f import (
        FeatureEngineeringWrapper,
        AutoKillWrapper,
        FrameStackingWrapper
    )
except ImportError:
    print("2")
    # Si tu lances depuis la racine du dossier template
    from stk_actor.wrappers import (
        FeatureEngineeringWrapper,
        FrameStackingWrapper
    )




def test_wrappers_robustness():
    print("\n⚔️  TEST DE ROBUSTESSE : MULTI-FULL-V0 vs SIMPLE-V0 ⚔️")
    
    # 1. On lance l'environnement "Serveur" (Multi-Full)
    # Le serveur utilise souvent max_paths ou d'autres paramètres
    env_multi = gym.make(
        "supertuxkart/multi-full-v0", 
        render_mode=None, 
        num_kart=2, 
        agents=[AgentSpec(name="TestAgent", use_ai=False)]
    )
    
    print("✅ Environment multi-full-v0 chargé.")
    
    obs_multi, _ = env_multi.reset()
    
    # Le serveur renvoie un dict de dicts : {'0': {...}, '1': {...}}
    # On simule l'extraction de l'agent 0 comme le fait le wrapper MonoAgent
    raw_obs = obs_multi['0'] 
    
    print(f"ℹ️  Clés brutes reçues de multi-full-v0 : {len(raw_obs.keys())}")
    # print(sorted(raw_obs.keys())) # Décommenter pour voir les intrus potentiels
    
    # 2. Création d'un Dummy Env pour appliquer tes wrappers
    # On doit tricher un peu car tes wrappers s'attendent à un env.step()
    # On va wrapper manuellement l'environnement multi en simulant un mono
    
    class PseudoMonoEnv(gym.Wrapper):
        def __init__(self, env):
            super().__init__(env)
            self.observation_space = env.observation_space['0'] # On prend l'espace de l'agent 0
            self.action_space = env.action_space['0']
            
        def reset(self, **kwargs):
            obs, info = self.env.reset(**kwargs)
            return obs['0'], info # On extrait direct
            
        def step(self, action):
            # Le wrapper attend une action simple, on la met dans un dict pour multi
            actions = {'0': action}
            obs, reward, terminated, truncated, info = self.env.step(actions)
            return obs['0'], reward['0'], terminated, truncated, info

    # On wrap l'environnement multi pour qu'il ressemble à un simple
    env_simulated = PseudoMonoEnv(env_multi)
    
    # 3. Application de TES Wrappers
    env_wrapped = FeatureEngineeringWrapper(env_simulated)
    env_wrapped = DiscreteActionWrapper(env_wrapped) # C'est LUI le test critique
    env_wrapped = FrameStackingWrapper(env_wrapped, n_stack=4)
    
    print("\n🏗️  Application des wrappers terminée.")
    print(f"📏 Shape finale de l'observation : {env_wrapped.observation_space.shape}")
    
    # 4. Vérification
    obs, _ = env_wrapped.reset()
    print(f"Output vector shape: {obs.shape}")
    
    # Test d'une step
    action = env_wrapped.action_space.sample()
    obs, reward, terminated, truncated, info = env_wrapped.step(action)
    
    expected_size = 112 * 4 # 112 features * 4 stack = 448
    if obs.shape[0] == expected_size:
        print(f"\n✅ SUCCÈS ! Taille {obs.shape[0]} correspond à l'attendu ({expected_size}).")
        print("L'agent ne crashera pas sur le serveur.")
    else:
        print(f"\n⚠️  ATTENTION ! Taille {obs.shape[0]} != Attendu {expected_size}.")
        print("Vérifie la liste self.keep_keys dans DiscreteActionWrapper.")

    env_multi.close()

if __name__ == "__main__":
    test_wrappers_robustness()

1
2

⚔️  TEST DE ROBUSTESSE : MULTI-FULL-V0 vs SIMPLE-V0 ⚔️
✅ Environment multi-full-v0 chargé.
ℹ️  Clés brutes reçues de multi-full-v0 : 22
🔒 DiscreteActionWrapper: Taille FORCÉE et FIXÉE à 112

🏗️  Application des wrappers terminée.
📏 Shape finale de l'observation : (448,)
Output vector shape: (448,)


IndexError: invalid index to scalar variable.

In [40]:
import gymnasium as gym
from pystk2_gymnasium import AgentSpec
import numpy as np
import sys
import os

# Assure-toi que les wrappers importés sont bien ceux mis à jour
try:
    from stk_actor.wrappers import (
        FeatureEngineeringWrapper,
        # DiscreteActionWrapper, # <--- Celui qu'on vient de modifier
        FrameStackingWrapper
    )
    print("Wrappers importés.")
except ImportError:
    # Fallback pour le notebook local
    from stk_actor.wrappers_f import *

def test_wrappers_robustness():
    print("\n⚔️  TEST DE ROBUSTESSE : MULTI-FULL-V0 vs SIMPLE-V0 ⚔️")
    
    # 1. Environment Serveur
    env_multi = gym.make(
        "supertuxkart/multi-full-v0", 
        render_mode=None, 
        num_kart=4, 
        agents=[AgentSpec(name="TestAgent", use_ai=False)]
    )
    
    print("✅ Environment multi-full-v0 chargé.")

    # 2. Wrapper PseudoMonoEnv (Corrigé pour le IndexError)
    class PseudoMonoEnv(gym.Wrapper):
        def __init__(self, env):
            super().__init__(env)
            # On prend l'espace de l'agent 0 s'il existe, sinon fallback
            self.observation_space = env.observation_space['0'] 
            self.action_space = env.action_space['0']
            
        def reset(self, **kwargs):
            obs, info = self.env.reset(**kwargs)
            return obs['0'], info
            
        def step(self, action):
            actions = {'0': action}
            obs, reward, terminated, truncated, info = self.env.step(actions)
            
            # --- CORRECTION DU CRASH INDEX ERROR ---
            # Dans certains cas, reward est déjà un float, pas un dict
            r = reward['0'] if isinstance(reward, dict) else float(reward)
            t = terminated['0'] if isinstance(terminated, dict) else bool(terminated)
            tr = truncated['0'] if isinstance(truncated, dict) else bool(truncated)
            
            return obs['0'], r, t, tr, info

    env_simulated = PseudoMonoEnv(env_multi)
    
    # 3. Application de tes wrappers
    print("Construction de la stack de wrappers...")
    env_wrapped = FeatureEngineeringWrapper(env_simulated)
    env_wrapped = DiscreteActionWrapper(env_wrapped) # Doit afficher ~112
    env_wrapped = FrameStackingWrapper(env_wrapped, n_stack=4)
    
    # 4. Vérification finale
    print("\n--- VÉRIFICATION FINALE ---")
    obs, _ = env_wrapped.reset()
    print(f"Shape finale Observation: {obs.shape}")
    
    # Test step
    action = env_wrapped.action_space.sample()
    obs, reward, terminated, truncated, info = env_wrapped.step(action)
    print("Step OK.")
    
    expected_size = 112 * 4 # 448
    if obs.shape[0] == expected_size:
        print(f"\n✅ SUCCÈS TOTAL ! Taille {obs.shape[0]} parfaite.")
        print("Tu peux PUSH sur le serveur les yeux fermés.")
    else:
        print(f"\n⚠️  ATTENTION : Taille {obs.shape[0]}. (Attendu : {expected_size})")
        print("Si l'écart est petit (ex: 452), c'est peut-être juste un changement de config mineur.")
        print("Si l'écart est énorme (>1000), le slicing ne marche pas.")

    env_multi.close()

if __name__ == "__main__":
    test_wrappers_robustness()

Wrappers importés.

⚔️  TEST DE ROBUSTESSE : MULTI-FULL-V0 vs SIMPLE-V0 ⚔️
✅ Environment multi-full-v0 chargé.
Construction de la stack de wrappers...
🔒 DiscreteActionWrapper: Taille FORCÉE et FIXÉE à 112

--- VÉRIFICATION FINALE ---
Shape finale Observation: (448,)
Step OK.

✅ SUCCÈS TOTAL ! Taille 448 parfaite.
Tu peux PUSH sur le serveur les yeux fermés.


---

In [3]:
import gymnasium as gym
import torch
import numpy as np
import os
import sys
from bbrl.workspace import Workspace
from pystk2_gymnasium import AgentSpec

# On ajoute le dossier courant au path pour trouver stk_actor
sys.path.append(os.getcwd())

def main():
    print("="*60)
    print("🚀 SIMULATION DU SERVEUR D'ÉVALUATION")
    print("="*60)

    # 1. VÉRIFICATION DES FICHIERS
    pth_path = os.path.join("stk_actor", "pystk_actor.pth")
    if not os.path.exists(pth_path):
        print(f"❌ ERREUR CRITIQUE: Le fichier {pth_path} est introuvable !")
        return

    # 2. IMPORT DU MODULE (Simulation de importlib)
    print("🔹 Chargement du module 'stk_actor.pystk_actor'...")
    try:
        import stk_actor.pystk_actor as player_module
    except ImportError as e:
        print(f"❌ ERREUR D'IMPORT : {e}")
        print("Vérifie que 'wrappers.py' est bien nommé et que l'import dans pystk_actor.py est correct.")
        return

    # 3. CRÉATION DE L'ENVIRONNEMENT (Multi-Full-v0)
    print("🔹 Création de l'environnement 'supertuxkart/multi-full-v0'...")
    # Le serveur utilise souvent max_paths pour limiter ou non, et num_kart >= 1
    base_env = gym.make(
        "supertuxkart/multi-full-v0",
        render_mode="human", # Mettre "human" si tu veux voir le jeu (mais ferme la fenêtre pour continuer)
        num_kart=4,
        agents=[AgentSpec(name="TestAgent", use_ai=False)]
    )

    # 4. ADAPTATEUR SERVEUR (Simule MonoAgentWrapperAdapter)
    # C'est ce bout de code qui transforme le dict {'0': ...} en observation simple pour tes wrappers
    class ServerAdapter(gym.Wrapper):
        def __init__(self, env):
            super().__init__(env)
            # On simule l'espace d'observation de l'agent 0
            self.observation_space = env.observation_space['0']
            self.action_space = env.action_space['0']
        
        def reset(self, **kwargs):
            obs, info = self.env.reset(**kwargs)
            return obs['0'], info
        
        def step(self, action):
            # Le wrapper sort une action simple, on la remet dans le format multi
            actions = {'0': action}
            obs, reward, terminated, truncated, info = self.env.step(actions)
            
            # Gestion robuste des retours (parfois scalaires, parfois dicts)
            r = reward['0'] if isinstance(reward, dict) else float(reward)
            t = terminated['0'] if isinstance(terminated, dict) else bool(terminated)
            tr = truncated['0'] if isinstance(truncated, dict) else bool(truncated)
            
            return obs['0'], r, t, tr, info

    env = ServerAdapter(base_env)

    # 5. APPLICATION DE TES WRAPPERS
    print("🔹 Application de tes wrappers...")
    try:
        wrappers = player_module.get_wrappers()
        for w in wrappers:
            env = w(env)
    except Exception as e:
        print(f"❌ ERREUR DANS TES WRAPPERS : {e}")
        import traceback
        traceback.print_exc()
        return

    print(f"   ✅ Wrappers appliqués. Shape finale: {env.observation_space.shape}")

    # 6. CHARGEMENT DE L'AGENT
    print("🔹 Chargement des poids et de l'acteur...")
    try:
        # Simulation du chargement sécurisé du serveur
        state_dict = torch.load(pth_path, map_location="cpu", weights_only=True)
        
        actor = player_module.get_actor(
            state_dict, 
            env.observation_space, 
            env.action_space
        )
        actor.eval() # Mode évaluation
    except Exception as e:
        print(f"❌ ERREUR CHARGEMENT ACTEUR : {e}")
        return

    # 7. BOUCLE DE COURSE (BBRL WORKSPACE)
    print("🔹 Lancement de la course (50 steps)...")
    workspace = Workspace()
    
    try:
        obs, _ = env.reset()
        t = 0
        done = False
        
        while not done and t < 400: # On teste sur 50 frames
            # A. Mettre l'observation dans le workspace
            # Ton acteur lit "env/env_obs", donc on met "env/env_obs"
            # Note: ParallelGymAgent fait ça automatiquement, ici on le fait à la main
            workspace.set("env/env_obs", t, torch.tensor(obs).unsqueeze(0)) # Unsqueeze pour le batch dim
            
            # B. Exécuter l'agent
            actor(workspace, t=t)
            
            # C. Récupérer l'action
            # ArgmaxActor écrit dans "action"
            action_tensor = workspace.get("action", t)
            action = action_tensor[0].item() # On récupère l'entier
            
            # D. Step Environnement
            obs, reward, terminated, truncated, info = env.step(action)
            
            done = terminated or truncated
            t += 1
            
            if t % 10 == 0:
                print(f"   Step {t}: Action={action}, Reward={reward:.4f}")

    except RuntimeError as e:
        print(f"❌ CRASH RUNTIME (Probablement une erreur de shape) : {e}")
        print("💡 Conseil : Vérifie que la taille de sortie de DiscreteActionWrapper correspond EXATEMENT à l'entrée de ton réseau.")
        return
    except Exception as e:
        print(f"❌ ERREUR D'EXÉCUTION : {e}")
        import traceback
        traceback.print_exc()
        return
    finally:
        env.close()

    print("="*60)
    print("✅ TEST RÉUSSI ! TON AGENT EST PRÊT POUR LE SERVEUR.")
    print("="*60)

if __name__ == "__main__":
    main()

🚀 SIMULATION DU SERVEUR D'ÉVALUATION
🔹 Chargement du module 'stk_actor.pystk_actor'...
🔹 Création de l'environnement 'supertuxkart/multi-full-v0'...
🔹 Application de tes wrappers...
   ✅ Wrappers appliqués. Shape finale: (448,)
🔹 Chargement des poids et de l'acteur...
❌ ERREUR CHARGEMENT ACTEUR : Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if you trust the source of the checkpoint. 
	(1) Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy._core.multiarray._reconstruct was not an allowed global by default. Please use `torch.serialization.add_safe_globals([_reconstruct])` to allowlist this global if you trust this class/fun

In [2]:
import gymnasium as gym
import pystk2_gymnasium # Indispensable
import torch
import numpy as np
import os
import sys
from bbrl.workspace import Workspace

# --- CONFIGURATION ---
# On force 2 karts pour tester ton fix (112 inputs au lieu de 154)
# Si ton code marche avec num_kart=2, il marchera sur le serveur !
NUM_KARTS_TEST = 2 
RENDER_MODE = "human" # "human" pour voir, None pour aller vite

def run_server_simulation():
    print("--- 🕵️ SIMULATION DU SERVEUR D'EVALUATION 🕵️ ---")

    # 1. Vérification des fichiers
    if not os.path.exists("stk_actor/pystk_actor.pth"):
        print("❌ ERREUR : Le fichier 'stk_actor/pystk_actor.pth' est introuvable.")
        return

    # 2. Importation de TON code (comme le fait le serveur)
    try:
        from stk_actor.pystk_actor import get_actor, get_wrappers, env_name
        print(f"✅ Importation réussie depuis stk_actor (Env: {env_name})")
    except ImportError as e:
        print(f"❌ ERREUR IMPORT : {e}")
        print("Vérifie que tu es à la racine du projet.")
        return

    # 3. Création de l'environnement (avec le piège des 2 karts)
    try:
        print(f"🔹 Création de l'environnement avec {NUM_KARTS_TEST} karts...")
        env = gym.make(
            env_name, 
            render_mode=RENDER_MODE,
            num_kart=4 # <--- LE TEST ULTIME POUR LE PADDING
        )
    except Exception as e:
        print(f"❌ Erreur Gym : {e}")
        return

    # 4. Application de TES wrappers
    wrappers = get_wrappers()
    for w in wrappers:
        env = w(env)
    
    # 5. Chargement de l'Agent
    print("🔹 Chargement des poids...")
    try:
        state_dict = torch.load("stk_actor/pystk_actor.pth", map_location="cpu")
        actor = get_actor(state_dict, env.observation_space, env.action_space)
        print("✅ Agent chargé avec succès.")
    except Exception as e:
        print(f"❌ Erreur chargement agent : {e}")
        return

    # 6. BOUCLE DE SIMULATION (Style BBRL)
    print("\n🟢 DÉBUT DE LA COURSE")
    print("Si ça ne plante pas ici, ton 'Auto-Padding' fonctionne !")
    
    obs, _ = env.reset()
    workspace = Workspace()
    t = 0
    done = False
    
    try:
        while not done:
            # --- CE QUE FAIT LE SERVEUR ---
            
            # A. Met l'observation dans le Workspace
            # Conversion en Tensor + Batch dimension (1, InputSize)
            obs_tensor = torch.tensor(obs, dtype=torch.float32)
            if obs_tensor.dim() == 1:
                obs_tensor = obs_tensor.unsqueeze(0)
            
            # Debug pour toi : Voir la taille réelle reçue
            if t == 0:
                print(f"ℹ️  Taille reçue par l'acteur : {obs_tensor.shape}")
                if obs_tensor.shape[1] == 112:
                    print("⚠️  ATTENTION : On reçoit 112 (2 karts). L'acteur doit padder à 154 !")

            workspace.set("env/env_obs", t, obs_tensor)
            
            # B. L'Acteur agit
            # C'est ici que ton code dans actors.py est exécuté
            actor(workspace, t=t)
            
            # C. Récupère l'action
            action = workspace.get("action", t)
            action_np = action.squeeze(0).numpy()
            
            # D. Joue l'action
            obs, reward, terminated, truncated, _ = env.step(action_np)
            done = terminated or truncated
            t += 1
            
    except RuntimeError as e:
        print(f"\n❌ CRASH RUNTIME : {e}")
        if "size of tensor a" in str(e):
            print("👉 C'est l'erreur de taille ! Ton 'Auto-Padding' dans actors.py ne marche pas ou n'est pas sauvegardé.")
    except Exception as e:
        print(f"\n❌ ERREUR : {e}")
    finally:
        env.close()
        print("\n🏁 Fin de la simulation.")

if __name__ == "__main__":
    run_server_simulation()

--- 🕵️ SIMULATION DU SERVEUR D'EVALUATION 🕵️ ---
✅ Importation réussie depuis stk_actor (Env: supertuxkart/simple-v0)
🔹 Création de l'environnement avec 2 karts...
🔹 Chargement des poids...
❌ Erreur chargement agent : cannot assign 'numpy.ndarray' object to buffer 'obs_mean' (torch Tensor or None required)


C:\Users\PC PRO DZ\AppData\Local\Temp\ipykernel_22124\2037378846.py:52: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load("stk_actor/pystk_actor.pth", ma